In [17]:
import dagshub
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

# свързване с dagshub
dagshub.init(repo_owner='cvetygeorg', repo_name='my-repo', mlflow=True)
# Създаване/избор на експеримент
mlflow.set_experiment("students-online-CatBoost")

# Данни
df = pd.read_csv('students-online.csv')
X = df.drop(['Satisfaction', 'ID'], axis=1)
y = df['Satisfaction']
categorical_features_indices = [0, 2, 3, 7]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Параметри на модела
iters = 100
lr = 0.1
d = 6

with mlflow.start_run(run_name="experiment_run_100"):
    # Създаване и обучение на модел
    model = CatBoostClassifier(iterations=iters, learning_rate=lr, depth=d, verbose=False)
    model.fit(X_train, y_train, cat_features=categorical_features_indices)

    # Прогнози
    y_pred = model.predict(X_test)

    # Мерки
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    # Логване на параметри
    mlflow.log_param("iterations", iters)
    mlflow.log_param("learning_rate", lr)
    mlflow.log_param("depth", d)
    
    # Логване на метрики
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_macro", f1)

    # Запис на модела като артефакт
    mlflow.sklearn.log_model(model, name = "CatBoost_model")

    print("Run completed")
    print("accuracy =", accuracy)
    print("f1_macro =", f1)

Initialized MLflow to track repo "cvetygeorg/my-repo"

Repository cvetygeorg/my-repo initialized!

2026/04/10 20:08:54 INFO mlflow.tracking.fluent: Experiment with name 'students-online-CatBoost' does not exist. Creating a new experiment.
2026/04/10 20:09:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run completed
accuracy = 0.75
f1_macro = 0.6666666666666666
🏃 View run experiment_run_100 at: https://dagshub.com/cvetygeorg/my-repo.mlflow/#/experiments/0/runs/3a7148ec046749a7be33f40d1d490910
🧪 View experiment at: https://dagshub.com/cvetygeorg/my-repo.mlflow/#/experiments/0
